# Power Grid Transient Stability — v2

## Métrica: Grid Risk Weighted Error (GRWE)
- Penalty=2.5 cuando `y_true > Q75 AND y_pred < y_true` → costo 6.25x
- Pesos por bus: 1.0, 1.2, 1.4, 1.6, 1.8, 2.0

## Arquitectura
1. **Feature engineering** físicamente motivado (sin PCA — features ya son ortogonales)
2. **CatBoost** por bus con bias correction asimétrico en OOF
3. **NN multi-output** con shared trunk + 6 heads y GRWE loss exacto
4. **Ensemble** CB + NN con pesos optimizados por GRWE
5. **Bias correction** global post-ensemble

In [1]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
pip('lightgbm'); pip('catboost'); pip('xgboost')

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from scipy.optimize import minimize

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import catboost as cb
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [3]:
PLATFORM_DATA_DIR = Path('dataset/public')
if PLATFORM_DATA_DIR.exists():
    TRAIN_PATH  = PLATFORM_DATA_DIR / 'train.csv'
    TEST_PATH   = PLATFORM_DATA_DIR / 'test.csv'
    SUBMIT_PATH = Path('working/submission.csv')
    Path('working').mkdir(exist_ok=True)
else:
    from google.colab import files
    files.upload()
    TRAIN_PATH  = Path('train.csv')
    TEST_PATH   = Path('test.csv')
    SUBMIT_PATH = Path('submission.csv')

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print(f'Train: {train.shape} | Test: {test.shape}')

Saving train.csv to train.csv
Saving test.csv to test.csv
Saving sample_submission.csv to sample_submission.csv
Train: (30000, 20) | Test: (25000, 14)


In [4]:
TARGETS   = [f'bus_risk_{i}' for i in range(1, 7)]
BUS_W     = np.array([1.0, 1.2, 1.4, 1.6, 1.8, 2.0])
ORIG_FEAT = [c for c in train.columns if c not in ['id'] + TARGETS]

SEED   = 42
MODE   = 'precision'  # 'quickrun' | 'precision'
N_FOLDS = 3  if MODE == 'quickrun' else 5
SEEDS   = [42] if MODE == 'quickrun' else [42, 7, 123, 0]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Mode: {MODE} | Folds: {N_FOLDS} | Seeds: {SEEDS} | Device: {DEVICE}')

# Q75 calculado sobre train — usado para penalty en GRWE
Q75     = np.array([train[t].quantile(0.75) for t in TARGETS])
Q75_dict = {t: train[t].quantile(0.75) for t in TARGETS}
print('Q75:', np.round(Q75, 4))

Mode: precision | Folds: 5 | Seeds: [42, 7, 123, 0] | Device: cuda
Q75: [2.5523 2.9487 3.3483 3.7474 4.1132 4.5162]


In [5]:
# ── GRWE metric ────────────────────────────────────────────────────────────────
def grwe_full(y_true, y_pred):
    """
    y_true, y_pred: (n, 6) numpy arrays
    Returns scalar GRWE.
    """
    rmses = []
    for k in range(6):
        yt      = y_true[:, k]
        yp      = y_pred[:, k]
        penalty = np.where((yt > Q75[k]) & (yp < yt), 2.5, 1.0)
        rmses.append(np.sqrt(np.mean((penalty * (yp - yt))**2)))
    return np.dot(BUS_W, rmses) / BUS_W.sum()


def grwe_single(y_true, y_pred, k):
    """GRWE para un bus (índice 0-5)."""
    penalty = np.where((y_true > Q75[k]) & (y_pred < y_true), 2.5, 1.0)
    return np.sqrt(np.mean((penalty * (y_pred - y_true))**2))


print('GRWE metric defined.')

GRWE metric defined.


In [6]:
# ── Feature engineering ────────────────────────────────────────────────────────
def build_features(df):
    X = df[ORIG_FEAT].copy()

    # Renovables
    X['renewable_share']     = X['wind_share'] + X['solar_share']
    X['wind_x_contin']       = X['wind_share'] * X['contingency_index']
    X['solar_x_contin']      = X['solar_share'] * X['contingency_index']
    X['renewable_x_contin']  = X['renewable_share'] * X['contingency_index']
    X['wind_sq']             = X['wind_share'] ** 2
    X['renewable_sq']        = X['renewable_share'] ** 2
    X['inertia_x_wind']      = X['inertia_constant'] * X['wind_share']

    # Carga y saturación
    X['apparent_power']      = np.sqrt(X['load_mw']**2 + X['reactive_power_mvar']**2)
    X['power_factor']        = X['load_mw'] / (X['apparent_power'] + 1e-9)
    X['load_x_line']         = X['load_mw'] * X['line_utilization']
    X['load_x_react']        = X['load_mw'] * X['reactive_power_mvar']
    X['load_fluct_x_load']   = X['load_fluctuation_rate'] * X['load_mw']

    # Estabilidad y reservas
    X['stability_x_inertia'] = X['grid_stability_factor'] * X['inertia_constant']
    X['reserve_margin']      = X['reactive_reserve'] / (X['reactive_power_mvar'] + 1e-9)
    X['reserve_x_stability'] = X['reactive_reserve'] * X['grid_stability_factor']
    X['contin_x_fluct']      = X['contingency_index'] * X['load_fluctuation_rate']

    # Frecuencia y tensión
    X['freq_x_inertia']      = X['frequency_hz'] * X['inertia_constant']
    X['voltage_x_reserve']   = X['voltage_pu'] * X['reactive_reserve']

    # Flujo de red
    X['tie_x_contin']        = X['tie_line_flow'] * X['contingency_index']
    X['tie_x_line_util']     = X['tie_line_flow'] * X['line_utilization']

    # Score de riesgo agregado físicamente motivado
    X['risk_score'] = (
        X['renewable_share'] * X['contingency_index'] * X['load_fluctuation_rate']
        / (X['inertia_constant'] * X['reactive_reserve'] + 1e-9)
    )

    # Interacciones triples — escenario crítico exacto
    X['wind_x_contin_x_fluct']    = X['wind_share'] * X['contingency_index'] * X['load_fluctuation_rate']
    X['renew_x_contin_x_fluct']   = X['renewable_share'] * X['contingency_index'] * X['load_fluctuation_rate']
    X['load_x_line_x_contin']     = X['load_mw'] * X['line_utilization'] * X['contingency_index']
    X['inertia_x_wind_x_contin']  = X['inertia_constant'] * X['wind_share'] * X['contingency_index']
    X['reserve_x_inertia_x_wind'] = X['reactive_reserve'] * X['inertia_constant'] * X['wind_share']

    # Ratios de stress — cuánto stress hay vs cuánta capacidad de amortiguación
    X['stress_ratio']    = (X['load_mw'] * X['contingency_index']) / (X['inertia_constant'] * X['reactive_reserve'] + 1e-9)
    X['renewable_ratio'] = X['renewable_share'] / (X['inertia_constant'] + 1e-9)
    X['fluct_ratio']     = X['load_fluctuation_rate'] / (X['grid_stability_factor'] + 1e-9)

    return X


X_train_fe = build_features(train)
X_test_fe  = build_features(test)
FEAT_COLS  = X_train_fe.columns.tolist()

X_train = X_train_fe.values.astype(np.float32)
X_test  = X_test_fe.values.astype(np.float32)
Y_train = train[TARGETS].values.astype(np.float32)

print(f'Features: {len(FEAT_COLS)} | Train: {X_train.shape} | Test: {X_test.shape}')

Features: 42 | Train: (30000, 42) | Test: (25000, 42)


In [7]:
# ════════════════════════════════════════════════════════════════════════
# MODELO 1: CatBoost por bus
# ════════════════════════════════════════════════════════════════════════
cb_params = dict(
    loss_function = 'RMSE',
    learning_rate = 0.05,
    iterations    = 500,
    depth=6, l2_leaf_reg=3.0,
    subsample=0.8,
    task_type='CPU',
    verbose=False,
)

oof_cb   = np.zeros((len(train), 6))
test_cb  = np.zeros((len(test),  6))

for k, bus in enumerate(TARGETS):
    y_k = Y_train[:, k]
    print(f'  CB {bus}...', end=' ')
    for seed in SEEDS:
        kf   = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        oof_s = np.zeros(len(train))
        test_s = np.zeros(len(test))
        for tr, va in kf.split(X_train):
            m = cb.CatBoostRegressor(**{**cb_params, 'random_seed': seed})
            m.fit(X_train[tr], y_k[tr],
                  eval_set=(X_train[va], y_k[va]),
                  early_stopping_rounds=50)
            oof_s[va] = m.predict(X_train[va])
            test_s   += m.predict(X_test) / N_FOLDS
        oof_cb[:, k]  += oof_s  / len(SEEDS)
        test_cb[:, k] += test_s / len(SEEDS)
    grwe_k = grwe_single(y_k, oof_cb[:, k], k)
    print(f'asym-RMSE={grwe_k:.4f}')

print(f'\nCB OOF GRWE: {grwe_full(Y_train, oof_cb):.4f}')

  CB bus_risk_1... asym-RMSE=0.1314
  CB bus_risk_2... asym-RMSE=0.1504
  CB bus_risk_3... asym-RMSE=0.1738
  CB bus_risk_4... asym-RMSE=0.1970
  CB bus_risk_5... asym-RMSE=0.2155
  CB bus_risk_6... asym-RMSE=0.2383

CB OOF GRWE: 0.1928


In [8]:
# ════════════════════════════════════════════════════════════════════════
# MODELO 2: NN multi-output con GRWE loss exacto
# ════════════════════════════════════════════════════════════════════════

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.05):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.net(x))


class GridNet(nn.Module):
    """
    Shared trunk captura estructura común a todos los buses (r>0.95 entre targets).
    6 heads independientes permiten especialización por bus.
    """
    def __init__(self, n_features, hidden=512, n_blocks=4, dropout=0.05):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(n_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(),
            nn.Dropout(dropout),
            *[ResBlock(hidden, dropout) for _ in range(n_blocks)]
        )
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, 128), nn.SiLU(),
                nn.Dropout(dropout / 2),
                nn.Linear(128, 1)
            ) for _ in range(6)
        ])

    def forward(self, x):
        h = self.trunk(x)
        return torch.cat([head(h) for head in self.heads], dim=1)  # (batch, 6)


def grwe_loss(y_pred, y_true, q75_tensor, bus_weights, penalty=3.5):
    """
    GRWE loss exacto (differentiable).
    y_pred, y_true: (batch, 6)
    """
    # Penalty factor: hard threshold no es diferenciable, usamos soft aproximation
    # Sigmoid suavizado para la condición y_true > Q75 AND y_pred < y_true
    is_high  = torch.sigmoid((y_true - q75_tensor) * 10)      # ~1 si y_true > Q75
    is_under = torch.sigmoid((y_true - y_pred) * 10)           # ~1 si y_pred < y_true
    p        = 1.0 + (penalty - 1.0) * is_high * is_under     # suave entre 1 y 2.5

    sq_err   = (p * (y_pred - y_true)) ** 2                    # (batch, 6)
    rmse_k   = torch.sqrt(sq_err.mean(dim=0) + 1e-8)           # (6,)
    grwe     = (bus_weights * rmse_k).sum() / bus_weights.sum()
    return grwe


def run_nn_cv(X_tr_all, Y_tr_all, X_te, seeds,
              epochs=150 if MODE == 'quickrun' else 400,
              lr=3e-4, hidden=256, n_blocks=3, dropout=0.1):

    oof_nn  = np.zeros((len(X_tr_all), 6))
    test_nn = np.zeros((len(X_te), 6))

    q75_t = torch.tensor(Q75, dtype=torch.float32).to(DEVICE)
    bw_t  = torch.tensor(BUS_W, dtype=torch.float32).to(DEVICE)

    for seed in seeds:
        torch.manual_seed(seed)
        kf     = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        oof_s  = np.zeros((len(X_tr_all), 6))
        test_s = np.zeros((len(X_te), 6))

        for fold, (tr, va) in enumerate(kf.split(X_tr_all)):
            sc    = StandardScaler().fit(X_tr_all[tr])
            Xtr   = torch.tensor(sc.transform(X_tr_all[tr]), dtype=torch.float32)
            Xva   = torch.tensor(sc.transform(X_tr_all[va]), dtype=torch.float32)
            Xte   = torch.tensor(sc.transform(X_te),         dtype=torch.float32)
            Ytr   = torch.tensor(Y_tr_all[tr], dtype=torch.float32)
            Yva   = torch.tensor(Y_tr_all[va], dtype=torch.float32)

            loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=512, shuffle=True)

            model = GridNet(Xtr.shape[1], hidden=hidden,
                            n_blocks=n_blocks, dropout=dropout).to(DEVICE)
            opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

            best_score, best_state, no_imp = np.inf, None, 0

            for epoch in range(epochs):
                model.train()
                for Xb, Yb in loader:
                    Xb, Yb = Xb.to(DEVICE), Yb.to(DEVICE)
                    loss = grwe_loss(model(Xb), Yb, q75_t, bw_t)
                    opt.zero_grad(); loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                sched.step()

                model.eval()
                with torch.no_grad():
                    pva   = model(Xva.to(DEVICE)).cpu().numpy()
                    score = grwe_full(Y_tr_all[va], pva)
                if score < best_score:
                    best_score = score
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                    no_imp = 0
                else:
                    no_imp += 1
                if no_imp >= 30:
                    break

            model.load_state_dict(best_state)
            model.eval()
            with torch.no_grad():
                oof_s[va] = model(Xva.to(DEVICE)).cpu().numpy()
                test_s   += model(Xte.to(DEVICE)).cpu().numpy() / N_FOLDS

        oof_nn  += oof_s  / len(seeds)
        test_nn += test_s / len(seeds)
        print(f'  [NN] Seed {seed}: OOF GRWE={grwe_full(Y_tr_all, oof_s):.4f}')

    return oof_nn, test_nn


print('GridNet defined.')

GridNet defined.


In [9]:
print('=== Training NN ===')
oof_nn, test_nn = run_nn_cv(X_train, Y_train, X_test, SEEDS)
print(f'\nNN OOF GRWE: {grwe_full(Y_train, oof_nn):.4f}')

=== Training NN ===
  [NN] Seed 42: OOF GRWE=0.1710
  [NN] Seed 7: OOF GRWE=0.1702
  [NN] Seed 123: OOF GRWE=0.1707
  [NN] Seed 0: OOF GRWE=0.1720

NN OOF GRWE: 0.1637


In [10]:
# ════════════════════════════════════════════════════════════════════════
# ENSEMBLE CB + NN con pesos optimizados por GRWE
# ════════════════════════════════════════════════════════════════════════
def ens_loss(w):
    w = np.abs(w) / np.abs(w).sum()
    blend = w[0] * oof_cb + w[1] * oof_nn
    return grwe_full(Y_train, blend)

res   = minimize(ens_loss, x0=[0.5, 0.5], method='Nelder-Mead',
                 options={'maxiter': 1000})
opt_w = np.abs(res.x) / np.abs(res.x).sum()

oof_ens  = opt_w[0] * oof_cb  + opt_w[1] * oof_nn
test_ens = opt_w[0] * test_cb + opt_w[1] * test_nn

print(f'Ensemble weights: CB={opt_w[0]:.3f} NN={opt_w[1]:.3f}')
print(f'Ensemble OOF GRWE: {grwe_full(Y_train, oof_ens):.4f}')

Ensemble weights: CB=0.188 NN=0.812
Ensemble OOF GRWE: 0.1623


In [11]:
# ════════════════════════════════════════════════════════════════════════
# BIAS CORRECTION por bus
# Dado el penalty asimétrico, el bias óptimo es positivo
# ════════════════════════════════════════════════════════════════════════
biases = np.zeros(6)
for k in range(6):
    def loss_b(b):
        return grwe_single(Y_train[:, k], oof_ens[:, k] + b[0], k)
    res = minimize(loss_b, x0=[0.0], method='Nelder-Mead',
                   options={'maxiter': 500, 'xatol': 1e-5})
    biases[k] = res.x[0]

oof_final  = oof_ens  + biases
test_final = test_ens + biases

print(f'Biases: {np.round(biases, 4)}')
print(f'OOF GRWE after bias: {grwe_full(Y_train, oof_final):.4f}')
print(f'\nPer-bus OOF asym-RMSE:')
for k, bus in enumerate(TARGETS):
    g = grwe_single(Y_train[:,k], oof_final[:,k], k)
    r = np.sqrt(mean_squared_error(Y_train[:,k], oof_final[:,k]))
    print(f'  {bus} (w={BUS_W[k]}): asym-RMSE={g:.4f} | RMSE={r:.4f} | bias={biases[k]:+.4f}')

Biases: [0.0083 0.005  0.0069 0.0071 0.0071 0.0085]
OOF GRWE after bias: 0.1620

Per-bus OOF asym-RMSE:
  bus_risk_1 (w=1.0): asym-RMSE=0.1104 | RMSE=0.0965 | bias=+0.0083
  bus_risk_2 (w=1.2): asym-RMSE=0.1277 | RMSE=0.1123 | bias=+0.0050
  bus_risk_3 (w=1.4): asym-RMSE=0.1459 | RMSE=0.1284 | bias=+0.0069
  bus_risk_4 (w=1.6): asym-RMSE=0.1638 | RMSE=0.1443 | bias=+0.0071
  bus_risk_5 (w=1.8): asym-RMSE=0.1820 | RMSE=0.1604 | bias=+0.0071
  bus_risk_6 (w=2.0): asym-RMSE=0.2003 | RMSE=0.1767 | bias=+0.0085


In [12]:
# ── Submission ─────────────────────────────────────────────────────────────────
submission = pd.DataFrame({'id': test['id']})
for k, bus in enumerate(TARGETS):
    submission[bus] = test_final[:, k]

submission.to_csv(SUBMIT_PATH, index=False)
print(f'Saved: {SUBMIT_PATH}')
print(submission[TARGETS].describe().T[['mean','std','min','max']].round(4).to_string())

Saved: submission.csv
              mean     std     min     max
bus_risk_1  2.3433  0.4764  0.8843  4.1097
bus_risk_2  2.7070  0.5898  1.0235  4.8653
bus_risk_3  3.0698  0.6359  1.1446  5.4801
bus_risk_4  3.4286  0.7165  1.3311  6.0548
bus_risk_5  3.7720  0.8065  1.3943  6.7235
bus_risk_6  4.1348  0.8927  1.5194  7.3708
